# Initializing the environment

In [ ]:
from gymnasium import make
from samples.llm_interface import OGPT4Interfacer
import os
import random
from natural20.gym.tools import compute_available_moves
os.environ["OPENAI_API_KEY"] = ""

MAX_EPISODES = 5
n_player1 = 2
n_player2 = 2
# Initialize the environment
# env = make("dndenv-v0", root_path="templates", render_mode="ansi")
all_classes = ['halfling_rogue.yml', 'high_elf_fighter.yml']
all_players = ["Alysha", "Bernard", "Cedric", "Didier", "Eric", "Francois", "Gertrude", "Heloise", "Isabelle"]
players = random.sample(all_players, n_player1 + n_player2)
a_player = [(random.choice(all_classes), player) for player in players[:n_player1]]
e_player = [(random.choice(all_classes), player) for player in players[n_player1:]]
env = make(
    "dndenv-v0",
    render_mode="ansi",
    map_file="maps/game_map.yml",
    show_logs=True,
    profiles=a_player,
    enemies=e_player,
    control_groups=["a","b"]
    )

envi = env.env.env
observation, info = env.reset(seed = random.randint(0,1000))




agents = {}
groups = env.env.env.battle.groups  
for character in env.env.env.battle.combat_order:
    gr = None
    for group_name, group in env.env.env.battle.groups.items():
        if character in group:
            gr = group_name
    if gr == None : print(f"Warning ! : Character {character.name} has no friends !!! (aka no groups attributed)")
    agents[character.name] = (OGPT4Interfacer(debug=False, explain=True, name=character.name), gr, character)

agents


/home/thomas/Documents/ENS/IAS/S2/Neural Networks/DandLLM/natural_20.py/samples/llm_interface.py:747: SyntaxWarning: invalid escape sequence '\d'
  regex = "\d"


Alysha rolled initiative d20(14) + 5 value 19.2
Bernard rolled initiative d20(15) + 5 value 20.2
Heloise rolled initiative d20(19) + 5 value 24.2
Isabelle rolled initiative d20(7) + 5 value 12.2
Alysha rolled initiative d20(16) + 5 value 21.2
Bernard rolled initiative d20(7) + 5 value 12.2
Heloise rolled initiative d20(5) + 5 value 10.2
Isabelle rolled initiative d20(15) + 5 value 20.2
Combat begins with 4 players.
Players: <p>Alysha (fighter-2) Team a</p>
<p>Bernard (rogue-2) Team a</p>
<p>Heloise (rogue-2) Team b</p>
<p>Isabelle (rogue-2) Team b</p>
======== Alysha starts their turn. ========
======== Alysha starts their turn. ========


/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:130: UserWarning: WARN: The obs returned by the `reset()` method was expecting a numpy array, actual type: <class 'list'>
  logger.warn(
/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/spaces/box.py:423: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


{'Alysha': (<samples.llm_interface.OGPT4Interfacer at 0x7f7a090a0590>,
  'a',
  Alysha),
 'Isabelle': (<samples.llm_interface.OGPT4Interfacer at 0x7f7a08f6e990>,
  'b',
  Isabelle),
 'Bernard': (<samples.llm_interface.OGPT4Interfacer at 0x7f7a08e7f250>,
  'a',
  Bernard),
 'Heloise': (<samples.llm_interface.OGPT4Interfacer at 0x7f7a08f67820>,
  'b',
  Heloise)}

In [2]:
class Experiment():
    def __init__(self, environment, dnd_environment, agents, debug=False):
        self.env = environment
        self.dnd_environment = dnd_environment
        self.agents = agents
        self.debug = debug
        self.backlog = []
        self.conversations = []
    
    def get_obs_inf(self, player):
        p_observation = self.dnd_environment.generate_observation(player)
        p_available_moves = compute_available_moves(self.dnd_environment.session, self.dnd_environment.map, player, self.dnd_environment.battle, self.dnd_environment.weapon_mappings, self.dnd_environment.spell_mappings)
        p_info = self.dnd_environment._info(p_available_moves, player)
        return p_observation, p_info
    
    def update_all_agents(self, agents_name, sender, content):
        for name in agents_name:
            self.agents[name][0].register_conversation(sender, content)
        self.conversations[-1].append((sender, content))

    def initiate_conversation(self, agents_name):
        for name in agents_name:
            self.agents[name][0].initiate_conversation()
        self.conversations.append([])

    def close_conversation(self, agents_name):
        for name in agents_name:
            obs, inf = self.get_obs_inf(self.agents[name][2])
            self.agents[name][0].close_conversation(obs, inf, self.dnd_environment.players)
        self.conversations[-1].append((None, "Conversation closed"))

    def run_conversation(self, sender, content):
        sender_gr = self.agents[sender][1]
        agent_in_the_conv = []
        for name, (_, gr, _) in self.agents.items():
            if sender_gr == gr and name != sender:
                agent_in_the_conv.append(name)
        agent_in_the_conv.append(sender)
        self.initiate_conversation(agent_in_the_conv)
        self.update_all_agents(agent_in_the_conv, sender, content)
        conv_alive = True
        conv_step = 0
        while conv_alive:
            conv_step += 1
            conv_alive = False
            for name in agent_in_the_conv:
                obs, inf = self.get_obs_inf(self.agents[name][2])
                action, descrition,  content = self.agents[name][0].select_action_for_state(obs, inf, self.dnd_environment.players, is_conversation=True)
                if action == -2:
                    self.update_all_agents(agent_in_the_conv, name, content)
                    conv_alive = True
                elif action != -3:
                    raise ValueError(f"A non conversation action {action} was used during a conversation by agent {name}")
        self.close_conversation(agent_in_the_conv)
    
    def step(self):
        current_agent, current_group, current_character = self.agents[self.dnd_environment.battle.current_turn().name]
        obs, inf = self.get_obs_inf(current_character)
        action, descrition, content = current_agent.select_action_for_state(obs, inf, self.dnd_environment.players)
        self.backlog.append((current_character.name, action, descrition))
        if action != -1:
            _, _, terminal, _, _ = self.env.step(action)
        else :
            self.backlog.append((current_character.name, -1, len(self.conversations)))
            self.run_conversation(sender=current_character.name, content=content)
            terminal = False
        return terminal
    
    def run_till_end(self, max_step= 30):
        done = False
        step = 0
        while not done and step < max_step:
            if self.debug: print(f"\n\n________________________________________________________________________________\n Starting step {step}:\n")
            done = self.step()
            step += 1

In [3]:
expe = Experiment(env, env.env.env, agents, debug=True)
expe.run_till_end(max_step=30)



________________________________________________________________________________
 Starting step 0:

prompt: -------------------------------
We are playing a game of Dungeons and Dragons 5th Edition. It is current your turn and you play 
as a hero character denoted by A (a level 2 fighter).Your health is at [100.]% specifically 24/24 
Your current conditions are:

You have as enemies :
 - Heloise denoted by H (a level 2 rogue).
    Their health is currently at 100.0%.
    Their current conditions are: 
 - Isabelle denoted by I (a level 2 rogue).
    Their health is currently at 100.0%.
    Their current conditions are: 
You must defeat all of them in order to win.

You are helped in that regard by your allies :
 - Bernard denoted by B (a level 2 rogue).
    Their health is currently at 100.0%.
    Their current conditions are: You have the following available actions and movement available:

Available movement: [30]ft
Available actions: 1
Bonus actions: 1
Reactions: 1


Note that disc

/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:130: UserWarning: WARN: The obs returned by the `step()` method was expecting a numpy array, actual type: <class 'list'>
  logger.warn(


{'action_id': 0, 'description': 'end my turn', 'explanation': "I have already taken my main action (attacking with the longbow) for this turn. I currently have no valid actions left other than ending my turn or using Second Wind, but since my HP is full, there's no reason to use Second Wind now. I will end my turn and allow my allies and enemies to continue the combat round."}
Summary prompt: -------------------------------
We are playing a game of Dungeons and Dragons 5th Edition. It is current your turn and you play 
as a hero character denoted by A (a level 2 fighter).Your health is at [100.]% specifically 24/24 
Your current conditions are:

You have as enemies :
 - Heloise denoted by H (a level 2 rogue).
    Their health is currently at 0.0%.
    Their current conditions are: Currently Prone, 
 - Isabelle denoted by I (a level 2 rogue).
    Their health is currently at 100.0%.
    Their current conditions are: 
You must defeat all of them in order to win.

You are helped in that r

In [4]:
expe.backlog

[('Alysha',
  (0, (0, 0), (-4, -3), 13, 1),
  'attack Isabelle with ranged weapon: longbow'),
 ('Alysha', (-1, (0, 0), (0, 0), 0, 0), 'end my turn'),
 ('Isabelle', (0, (0, 0), (-3, 2), 18, 1), 'Attack Bernard with shortbow.'),
 ('Isabelle', (5, (-1, -1), (0, 0), 0, 0), 'Dash as a bonus action.'),
 ('Isabelle', (-1, (0, 0), (0, 0), 0, 0), 'end my turn'),
 ('Bernard',
  (0, (0, 0), (3, -2), 18, 1),
  'attack Isabelle with ranged weapon: shortbow'),
 ('Bernard', (1, (-1, 0), (0, 0), 0, 0), 'move 5ft to the left'),
 ('Bernard',
  (5, (-1, -1), (0, 0), 0, 0),
  'Dash as bonus action to improve my positioning and possibly get closer to cover or a better angle for my next attack on Isabelle.'),
 ('Bernard', (-1, (0, 0), (0, 0), 0, 0), 'End my turn.'),
 ('Heloise', -1, 'communicate with my allies'),
 ('Heloise', -1, 0),
 ('Heloise', -1, 'communicate with my allies'),
 ('Heloise', -1, 1),
 ('Heloise',
  (-1, (0, 0), (0, 0), 0, 0),
  "End my turn because I'm at 0 HP and incapacitated, unable to 

In [6]:
env.env.env.players

[('a', 'H', Alysha, [15, 3]),
 ('a', 'H', Bernard, [8, 2]),
 ('b', 'E', Heloise, [2, 5]),
 ('b', 'E', Isabelle, [11, 0])]

In [8]:
agents["Spencer"][0].summary

'Combat has just started. I am Spencer (S), a level 2 rogue at full health (16/16). My current plan is to target Roger (R), the enemy rogue who is closest to me, with a ranged shortbow attack. This strategy allows me to apply early pressure and keep my distance, maximizing my safety and the effectiveness of my ranged capabilities. Mike (M), my ally, is also at full health and available to support. The initial plan is to weaken Roger before he can close into melee, possibly utilizing Sneak Attack if an opportunity arises.'

In [ ]:
# Select an action based on the initial state
current_agent, current_group, current_character = agents[env.env.env.battle.current_turn().name]

action = current_agent.select_action_for_state(p_observation, info, env.env.env.players)
print(f"Selected action: {action}")
# terminal = False
# episode = 0
# while not terminal and episode < MAX_EPISODES:
#     episode += 1
#     observation, reward, terminal, truncated, info = env.step(action)
#     if not terminal and not truncated:
#         print(env.render())
#         current_agent, current_group, current_character = agents[env.env.env.battle.current_turn().name]
#         action = current_agent.select_action_for_state(observation, info)
#         print(f"Selected action: {action}")

#     if terminal or truncated:
#         print(f"Reward: {reward}")
#         break
# action

prompt: -------------------------------
We are playing a game of Dungeons and Dragons 5th Edition. It is current your turn and you play 
as a hero character denoted by P (a level 2 rogue).Your health is at [100.]% specifically 16/16 
Your current conditions are:

You have as enemies :
 - Joe denoted by J (a level 2 wizard).
    Their health is currently at 100.0%.
    Their current conditions are: 
 - Roger denoted by R (a level 2 rogue).
    Their health is currently at 100.0%.
    Their current conditions are: 
You must defeat all of them in order to win.

You are helped in that regard by your allies :
 - Mike denoted by M (a level 2 fighter).
    Their health is currently at 100.0%.
    Their current conditions are: You have the following available actions and movement available:

Available movement: [25]ft
Available actions: 1
Bonus actions: 1
Reactions: 1



Here is a rough sketch of the map that considers line of sight to the enemy.
Here is the map:
____________
____________
____

/home/thomas/Documents/ENS/IAS/S2/Neural Networks/DandLLM/natural_20.py/samples/llm_interface.py:717: SyntaxWarning: invalid escape sequence '\d'
  regex = "\d"


KeyError: 'action'

In [7]:
info

{'available_moves': [(0, (0, 0), (-3, 4), 2, 1),
  (0, (0, 0), (-11, 2), 2, 1),
  (0, (0, 0), (-3, 4), 2, 1),
  (0, (0, 0), (-11, 2), 2, 1),
  (0, (0, 0), (-3, 4), 18, 1),
  (0, (0, 0), (-11, 2), 18, 1),
  (15, (-1, -1), (0, 0), 0, 0),
  (4, (-1, -1), (0, 0), 0, 0),
  (5, (-1, -1), (0, 0), 0, 0),
  (2, (-1, -1), (0, 0), 0, 0),
  (11, (-1, -1), (0, 0), 0, 0),
  (3, (-1, -1), (0, 0), 0, 0),
  (1, (-1, -1), (0, 0), 0, 0),
  (1, (-1, 0), (0, 0), 0, 0),
  (1, (-1, 1), (0, 0), 0, 0),
  (1, (0, -1), (0, 0), 0, 0),
  (1, (0, 1), (0, 0), 0, 0),
  (1, (1, -1), (0, 0), 0, 0),
  (1, (1, 0), (0, 0), 0, 0),
  (1, (1, 1), (0, 0), 0, 0),
  (10, (-1, -1), (0, 0), 0, 0),
  (14, (-1, -1), (0, 0), 0, 0),
  (16, (-1, -1), (0, 0), 0, 0),
  (16, (-1, -1), (0, 0), 0, 0),
  (-1, (0, 0), (0, 0), 0, 0)],
 'current_index': 0,
 'group': 'a',
 'round': 0,
 'health': 16,
 'max_health': 16,
 'weapon_mappings': {'unarmed': 0,
  'battleaxe': 1,
  'dagger': 2,
  'quarterstaff': 3,
  'sling': 4,
  'dart': 5,
  'greatclub

In [25]:
observation.keys()

dict_keys(['map', 'turn_info', 'conditions', 'health_pct', 'player_equipped', 'health_enemy', 'enemy_conditions', 'enemy_reactions', 'player_ac', 'enemy_ac', 'ability_info', 'player_type', 'enemy_type', 'spell_slots', 'movement', 'is_reaction'])

In [8]:
observation["health_enemy"]

array([1.])